# Tutorial 1: Descriptive Statistics and Exploratory Data Analysis

Welcome to the first tutorial in our statistical learning series! In this notebook, we'll explore fundamental concepts in descriptive statistics using the `statistics_lessons` package.

## Learning Objectives

By the end of this tutorial, you'll be able to:
- Calculate and interpret basic summary statistics
- Create and interpret visual representations of data distributions
- Identify outliers and assess data quality
- Apply these techniques to real datasets

## 1. Setup and Introduction

First, let's make sure we have our environment set up correctly.

In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Import our statistical utilities
from statistics_lessons.foundations.descriptive_stats import (
    summary_statistics, 
    median_absolute_deviation, 
    plot_histogram
)

# Set plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

## 2. Generating Sample Data

Before diving into real data, let's generate some synthetic data to understand the basics.

In [ ]:
# Set random seed for reproducibility
np.random.seed(42)

# Generate data from different distributions
normal_data = pd.Series(np.random.normal(loc=0, scale=1, size=1000), name="Normal")
skewed_data = pd.Series(np.random.exponential(scale=2, size=1000), name="Skewed")
bimodal_data = pd.Series(
    np.concatenate([
        np.random.normal(loc=-3, scale=1, size=500),
        np.random.normal(loc=3, scale=1, size=500)
    ]),
    name="Bimodal"
)

# Create a dataset with outliers
outlier_data = pd.Series(
    np.concatenate([
        np.random.normal(loc=5, scale=1, size=990),
        np.array([20, 21, 22, -10, -12, -15, 50, 55, 60, 100])
    ]),
    name="With Outliers"
)

## 3. Basic Summary Statistics

Let's calculate and interpret basic summary statistics for our datasets.

In [ ]:
# Calculate summary statistics for each dataset
datasets = [normal_data, skewed_data, bimodal_data, outlier_data]
results = []

for data in datasets:
    stats = summary_statistics(data)
    mad = median_absolute_deviation(data)
    stats['mad'] = mad
    stats['name'] = data.name
    results.append(stats)

# Convert to DataFrame for easy comparison
summary_df = pd.DataFrame(results).set_index('name')
summary_df

### Interactive Exercise 1: Interpreting Summary Statistics

🔍 **Examine the summary statistics table above and answer these questions:**

1. Which distribution has the largest gap between mean and median? Why?
2. How does the standard deviation compare to the MAD in the outlier dataset?
3. For which distribution type would you trust the mean as a measure of central tendency?

<details>
<summary>Click for answers</summary>

1. The skewed distribution shows the largest gap between mean and median. This occurs because the mean is sensitive to extreme values, while the median is not.

2. In the outlier dataset, the standard deviation is much larger than the MAD because standard deviation is heavily influenced by outliers, while MAD is robust against them.

3. The mean is most trustworthy for the normal distribution, as it's symmetric without heavy tails or outliers.
</details>

## 4. Visualizing Distributions

Let's visualize our datasets to better understand their characteristics.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for i, data in enumerate(datasets):
    # Use our utility function
    ax = plot_histogram(data)
    plt.close()  # Close the figure created by plot_histogram
    
    # Create a more detailed plot
    sns.histplot(data, kde=True, ax=axes[i])
    axes[i].set_title(f"Distribution: {data.name}")
    axes[i].axvline(data.mean(), color='red', linestyle='--', label='Mean')
    axes[i].axvline(data.median(), color='green', linestyle='--', label='Median')
    axes[i].legend()

plt.tight_layout()
plt.show()

### Interactive Exercise 2: Comparing Visualization Methods

Now, let's compare different visualization techniques for the same data.

In [ ]:
# Different ways to visualize the outlier dataset
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Histogram
sns.histplot(outlier_data, ax=axes[0, 0])
axes[0, 0].set_title("Histogram")

# Box plot
sns.boxplot(y=outlier_data, ax=axes[0, 1])
axes[0, 1].set_title("Box Plot")

# Violin plot
sns.violinplot(y=outlier_data, ax=axes[1, 0])
axes[1, 0].set_title("Violin Plot")

# Empirical CDF
from statsmodels.distributions.empirical_distribution import ECDF
ecdf = ECDF(outlier_data)
axes[1, 1].plot(ecdf.x, ecdf.y)
axes[1, 1].set_title("Empirical CDF")
axes[1, 1].set_xlabel("Value")
axes[1, 1].set_ylabel("Cumulative Probability")

plt.tight_layout()
plt.show()

🔍 **Consider the visualizations above and answer:**

1. Which plot best highlights the outliers?
2. Which visualization gives you the clearest sense of the central tendency?
3. What information can you extract from the ECDF that's harder to see in other plots?

<details>
<summary>Click for answers</summary>

1. The box plot most clearly highlights outliers, showing them as individual points outside the whiskers.

2. The violin plot provides the clearest visualization of central tendency and distribution shape, showing both the median and the probability density at different values.

3. The ECDF shows the cumulative distribution, making it easier to read percentiles and quantiles. It's also useful for comparing the empirical distribution to theoretical distributions.
</details>

## 5. Working with Real Data: Exploring the Iris Dataset

Now let's apply these techniques to a real dataset.

In [ ]:
from sklearn.datasets import load_iris

# Load the Iris dataset
iris = load_iris(as_frame=True)
iris_df = iris.data
iris_df['species'] = iris.target_names[iris.target]

# Display the first few rows
iris_df.head()

In [ ]:
# Calculate summary statistics for each feature
iris_stats = []

for column in iris_df.columns[:-1]:  # Exclude the species column
    stats = summary_statistics(iris_df[column])
    stats['feature'] = column
    iris_stats.append(stats)

pd.DataFrame(iris_stats).set_index('feature')

### Visualizing the Iris Dataset

In [ ]:
# Create pairplot to visualize relationships between features
sns.pairplot(iris_df, hue='species')
plt.suptitle('Iris Dataset: Pairwise Relationships', y=1.02)
plt.show()

In [ ]:
# Box plots by species
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for i, feature in enumerate(iris_df.columns[:-1]):
    sns.boxplot(x='species', y=feature, data=iris_df, ax=axes[i])
    axes[i].set_title(f"{feature} by Species")

plt.tight_layout()
plt.show()

## 6. Identifying Outliers

Let's explore methods for identifying outliers in our data.

In [ ]:
# Using Z-scores (standard deviations from the mean)
def find_outliers_zscore(data, threshold=3):
    """Find outliers using Z-scores."""
    z_scores = (data - data.mean()) / data.std()
    return data[abs(z_scores) > threshold]

# Using IQR (Interquartile Range)
def find_outliers_iqr(data, k=1.5):
    """Find outliers using the IQR method."""
    q1 = data.quantile(0.25)
    q3 = data.quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - k * iqr
    upper_bound = q3 + k * iqr
    return data[(data < lower_bound) | (data > upper_bound)]

# Using MAD (Median Absolute Deviation)
def find_outliers_mad(data, threshold=3.5):
    """Find outliers using the MAD method."""
    med = data.median()
    mad = median_absolute_deviation(data)
    modified_z_scores = 0.6745 * (data - med) / mad
    return data[abs(modified_z_scores) > threshold]

Let's apply these methods to our outlier dataset:

In [ ]:
# Find outliers using different methods
outliers_zscore = find_outliers_zscore(outlier_data)
outliers_iqr = find_outliers_iqr(outlier_data)
outliers_mad = find_outliers_mad(outlier_data)

print(f"Z-score method found {len(outliers_zscore)} outliers")
print(f"IQR method found {len(outliers_iqr)} outliers")
print(f"MAD method found {len(outliers_mad)} outliers")

# Compare the outliers found by each method
print("\nOutliers found by Z-score method:")
print(sorted(outliers_zscore.values))

print("\nOutliers found by IQR method:")
print(sorted(outliers_iqr.values))

print("\nOutliers found by MAD method:")
print(sorted(outliers_mad.values))

### Interactive Exercise 3: Outlier Detection

🔍 **Based on the results above, answer these questions:**

1. Which method detected the most outliers? Why?
2. Which method would you trust more for non-normally distributed data?
3. How might your choice of outlier detection method affect downstream analysis?

<details>
<summary>Click for answers</summary>

1. The IQR method typically detects the most outliers because it doesn't assume normality and uses quartiles which are less influenced by extreme values.

2. For non-normally distributed data, either the IQR method or the MAD method would be more appropriate than Z-scores, which assume normality. The MAD method is particularly robust against extreme values.

3. Choice of outlier detection method can significantly impact analysis:
   - Too sensitive methods might remove legitimate data points, reducing statistical power
   - Too lenient methods might leave problematic outliers that skew results
   - Different methods may be appropriate for different types of analyses (e.g., parametric vs. non-parametric tests)
</details>

## 7. Practice Exercise: Analyzing Housing Data

Now it's your turn to apply these techniques to analyze a housing dataset.

In [ ]:
# Load the Boston Housing dataset
from sklearn.datasets import fetch_california_housing

housing = fetch_california_housing(as_frame=True)
housing_df = housing.data
housing_df['PRICE'] = housing.target

# Select a random sample of 1000 homes to make visualization easier
housing_sample = housing_df.sample(1000, random_state=42)

# Your task: Analyze this dataset by:
# 1. Computing summary statistics for each feature
# 2. Creating appropriate visualizations
# 3. Identifying potential outliers
# 4. Drawing conclusions about the relationships between features and house prices

# Example starter code:
housing_sample.head()

<details>
<summary>Click for sample solution</summary>

In [ ]:
# 1. Summary statistics
housing_stats = housing_sample.describe()
print(housing_stats)

# 2. Visualizations
# Histograms of all features
housing_sample.hist(figsize=(15, 10), bins=30)
plt.tight_layout()
plt.show()

# Correlation heatmap
plt.figure(figsize=(10, 8))
correlation_matrix = housing_sample.corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Feature Correlations')
plt.show()

# Scatter plot of most correlated features with price
most_correlated = correlation_matrix['PRICE'].sort_values(ascending=False)[1:4].index
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for i, feature in enumerate(most_correlated):
    sns.scatterplot(x=feature, y='PRICE', data=housing_sample, ax=axes[i])
    axes[i].set_title(f'{feature} vs PRICE')

plt.tight_layout()
plt.show()

# 3. Outlier detection
# Using IQR method for each feature
outliers_by_feature = {}
for column in housing_sample.columns:
    outliers = find_outliers_iqr(housing_sample[column])
    outliers_by_feature[column] = len(outliers)

# Plot number of outliers by feature
plt.figure(figsize=(10, 6))
sns.barplot(x=list(outliers_by_feature.keys()), y=list(outliers_by_feature.values()))
plt.xticks(rotation=45)
plt.title('Number of Outliers by Feature (IQR Method)')
plt.tight_layout()
plt.show()

# 4. Conclusions
# - Features like MedInc (median income) show strong positive correlation with house prices
# - Population density seems to have minimal impact on housing prices
# - Several features have significant outliers that may need treatment in predictive models
# - The dataset shows clear relationships between geographical features (latitude/longitude) and prices

</details>

## 8. Summary and Key Takeaways

In this tutorial, we've covered:

1. **Basic summary statistics** and when to use different measures of central tendency and dispersion
2. **Visualization techniques** for understanding data distributions
3. **Outlier detection methods** and their relative strengths and weaknesses
4. **Application to real datasets** including the Iris dataset and housing data

### Next Steps

In the next tutorial, we'll explore:
- Probability distributions and sampling
- Inferential statistics and hypothesis testing
- Confidence intervals and their interpretation

## 9. Additional Resources

- [Pandas Documentation](https://pandas.pydata.org/docs/)
- [Seaborn Tutorial](https://seaborn.pydata.org/tutorial.html)
- [Outlier Detection Methods](https://towardsdatascience.com/5-ways-to-detect-outliers-that-every-data-scientist-should-know-python-code-70a54335a623)
- [Exploratory Data Analysis](https://r4ds.had.co.nz/exploratory-data-analysis.html) (from R for Data Science, but concepts apply to Python as well)